# 02. 역전파: 미분을 직접 계산하고 autograd와 비교

값이 알려진 직선 `y = 2x + 1`의 데이터로 시작한다. 가중치 `w`와 편향 `b`를 임의로 두고 평균제곱오차(MSE)를 줄여본다.

In [ ]:
import torch

assert torch.cuda.is_available(), "GPU가 활성화된 Jupyter 커널을 선택하세요."
device = torch.device("cuda")
x = torch.tensor([1., 2., 3., 4.], device=device)
target = 2 * x + 1
w = torch.tensor(-1., device=device, requires_grad=True)
b = torch.tensor(0., device=device, requires_grad=True)

prediction = w * x + b
loss = ((prediction - target) ** 2).mean()
print("prediction:", prediction.detach())
print("target:", target)
print("loss:", loss.item())

## 같은 미분을 두 방식으로 계산

MSE에서 각 예측값의 오차를 `prediction - target`이라고 놓는다. 미분 결과는 `dw = mean(2 * error * x)`, `db = mean(2 * error)`다. `loss.backward()`가 같은 값을 구하는지 비교한다.

In [ ]:
error = prediction.detach() - target
manual_dw = (2 * error * x).mean()
manual_db = (2 * error).mean()

loss.backward()
print("직접 계산 dw:", manual_dw.item(), "autograd dw:", w.grad.item())
print("직접 계산 db:", manual_db.item(), "autograd db:", b.grad.item())
torch.testing.assert_close(w.grad, manual_dw)
torch.testing.assert_close(b.grad, manual_db)

## 기울기를 이용해 실제로 가중치 수정

매번 새 예측값과 손실을 계산한 뒤 `backward()`를 실행한다. `no_grad()` 내부에서 가중치를 수정하고, 누적된 기울기를 0으로 초기화한다.

In [ ]:
w = torch.tensor(-1., device=device, requires_grad=True)
b = torch.tensor(0., device=device, requires_grad=True)
lr = 0.05
steps = 101

for step in range(steps):
    prediction = w * x + b
    loss = ((prediction - target) ** 2).mean()
    loss.backward()

    if step % 20 == 0:
        print(f"step={step:3d} loss={loss.item():.6f} w={w.item():.4f} b={b.item():.4f}")

    with torch.no_grad():
        w -= lr * w.grad
        b -= lr * b.grad
        w.grad.zero_()
        b.grad.zero_()

print("최종 w, b:", w.item(), b.item())

**직접 바꿔볼 것:** 학습률 `lr`을 0.005, 0.05, 0.5로 바꾸고 손실 변화를 관찰한다. `zero_()` 호출을 생략하면 기울기가 어떻게 누적되는지도 확인한다. 발산하면 커널을 재시작하고 원래 값으로 복구한다.